---
# Personalised Recommendation Associaton Rules
---

## Library Imports

In [30]:
# ============================================================================
# LIBRARY IMPORTS
# ============================================================================
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules
from itertools import combinations

# To suppress warnings during execution
import warnings

In [31]:
# Suppress all warnings while executing the code to keep output clean
warnings.filterwarnings('ignore')

---
# Association Rules | FP Growth Model
---

In [32]:
# Load data
borrowings = pd.read_csv("../../../data/processed/library_borrowings.csv")
borrowings['borrowing date'] = pd.to_datetime(borrowings['borrowing date'])

# Sort by reader and borrowing date
borrowings_sorted = borrowings.sort_values(['Reader_num', 'borrowing date'])

In [33]:
# Create itemsets - each reader's books form one transaction
itemsets_list = []
reader_ids = []

for reader_num, group in borrowings_sorted.groupby('Reader_num'):
    books = group['Title'].unique().tolist()
    if len(books) > 1:  # Only include readers with multiple books
        itemsets_list.append(set(books))
        reader_ids.append(reader_num)

print(f"Total transactions (readers): {len(itemsets_list)}\n")

# Create a dataframe to display transactions in an organized way
transactions_df = pd.DataFrame({
    'Reader_ID': reader_ids,
    'Books_Borrowed': [list(itemset) for itemset in itemsets_list],
    'Number_of_Books': [len(itemset) for itemset in itemsets_list]
})

print("="*150)
print("Transaction List (Each reader's books as one transaction):")
print("="*150)
display(transactions_df.head(15))
print(f"\n... and {len(transactions_df) - 15} more transactions")
print(f"\nStatistics:")
print(f"  Total transactions: {len(transactions_df)}")
print(f"  Min books per transaction: {transactions_df['Number_of_Books'].min()}")
print(f"  Max books per transaction: {transactions_df['Number_of_Books'].max()}")


Total transactions (readers): 119

Transaction List (Each reader's books as one transaction):


,Reader_ID,Books_Borrowed,Number_of_Books
0,1138472,"[C-C++ LA BIBLE DU PROGRAMMEUR : 1500 ASTUCES POUR TOUTES LES SITUATIONS, PARLER ET FAIRE UNE PRÉSENTATION EN PUBLIC EN TOUTE CONFIANCE : APPRENDRE]",2
1,11729923,"[PROGRAMMATION LINÉAIRE, RÉSEAUX INFORMATIQUES : RECUEIL DE SUJETS D'EXAMENS AVEC SOLUTIONS, RECHERCHE OPÉRATIONNELLE POUR INGÉNIEURS. 2]",3
2,11904505,"[PROBABILITES : RAPPELS DE COURS ET EXERCICES CORRIGES, FONCTIONS DE PLUSIEURS VARIABLES RELLES : IMITES,CONTINUITE,DIFFERENTIABILITÉ ET.... COURS DÉTAILLÉ ET EXERCICES RÉSOLUS]",2
3,12024,"[NEURAL NETWORKS AND DEEP LEARNING, HANDS ON MACHINE LEARNING WITH SCIKIT-LEARN, KERAS, AND TENSORFLOW : CONCEPTS, TOOLS, AND TECHNIQUES TO BUILD INTELLIGENT SYSTEMS, COMPUTER VISION ALGORITHMS AND APPLICATIONS : ALGORITHMS AND APPLICATIONS]",3
4,12225988,"[ALGEBRE 1 : RAPPELS DE COURS ET EXERCICES AVEC SOLUTIONS, COURS D'ALGÈBRE ET EXERCICES CORRIGÉS, ANALYSE MATHEMATIQUE : EXERCICES CORRIGES]",3
5,1244929,"[FONCTIONS RÉELLES D'UNE VARIABLE RÉELLE : DÉRIVABILITÉ, DÉRIVÉES, DÉVELOPPEMENTS LIMITÉS, DE MES CAHIERS D’ANALYSE.. INTÉGRALE DE RIEMANN, CALCUL DE PRIMITIVES, INTÉGRALES IMPROPRES, COURS DÉTAILLÉ ET EXERCICES RÉSOLUS]",2
6,1352515,"[PROBABILITÉS ET INTRODUCTION À LA STATISTIQUE, PROBABILITÉ : EXERCICES CORRIGÉS]",2
7,2336840,"[EXERCICES CORRIGÉS D'ALGÈBRE, ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING IN 2D-3D MEDICAL IMAGE PROCESSING]",2
8,32,"[CAMBRIDGE ENGLISH SKILLS REAL LISTENING AND SPEAKING 1, CAMBRIDGE ENGLISH SKILLS REAL LISTENING AND SPEAKING 2, COMPUTER ORGANIZATION AND ARCHITECTURE : DESIGNING FOR PERFORMANCE - GLOBAL EDITION]",3
9,33,"[TOUTES LES MATHÉMATIQUES, MP : 1RE PARTIE, LE KIT DE SURVIE, 2E PARTIE, DES CERISES SUR LE GÂTEAU, FONCTIONS DE PLUSIEURS VARIABLES RELLES : IMITES,CONTINUITE,DIFFERENTIABILITÉ ET.... COURS DÉTAILLÉ ET EXERCICES RÉSOLUS, MATHÉMATIQUES, EC 1RE ANNÉE : TOUT-EN-UN : COURS ET EXERCICES]",3



... and 104 more transactions

Statistics:
  Total transactions: 119
  Min books per transaction: 2
  Max books per transaction: 3


In [34]:
# Create binary matrix (One-Hot Encoding) for FP-Growth
# Rows = transactions (readers), Columns = books, Values = 1 or 0

# Get all unique books
all_books = borrowings['Title'].unique().tolist()

# Create binary matrix
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
transaction_matrix = mlb.fit_transform(itemsets_list)
df_transactions = pd.DataFrame(transaction_matrix, columns=mlb.classes_)


In [35]:
# Run FP-Growth algorithm
min_support = 0.02  # Books borrowed by at least 2% of readers

frequent_itemsets = fpgrowth(df_transactions, min_support=min_support, use_colnames=True)

print(f"Minimum support: {min_support:.0%}")
print(f"Total frequent itemsets found: {len(frequent_itemsets)}")
print(f"\n{'='*100}")
print(f"Top 20 frequent itemsets by support:")
print(f"{'='*100}\n")

# Sort by support descending
itemsets_sorted = frequent_itemsets.sort_values('support', ascending=False)

# Display with better formatting
for idx, row in itemsets_sorted.head(20).iterrows():
    itemset = ', '.join(sorted(list(row['itemsets'])))
    support_pct = row['support'] * 100
    count = int(row['support'] * len(itemsets_list))
    print(f"Itemset: {itemset}")
    print(f"  Support: {row['support']:.4f} ({support_pct:.2f}%) | Count: {count} readers")
    print()


Minimum support: 2%
Total frequent itemsets found: 26

Top 20 frequent itemsets by support:

Itemset: ALGEBRE 1 : RAPPELS DE COURS ET EXERCICES AVEC SOLUTIONS
  Support: 0.3193 (31.93%) | Count: 38 readers

Itemset: COURS D'ALGÈBRE ET EXERCICES CORRIGÉS
  Support: 0.2521 (25.21%) | Count: 30 readers

Itemset: FONCTIONS DE PLUSIEURS VARIABLES RELLES : IMITES,CONTINUITE,DIFFERENTIABILITÉ ET.... COURS DÉTAILLÉ ET EXERCICES RÉSOLUS
  Support: 0.2101 (21.01%) | Count: 25 readers

Itemset: ALGEBRE 1 : RAPPELS DE COURS ET EXERCICES AVEC SOLUTIONS, COURS D'ALGÈBRE ET EXERCICES CORRIGÉS
  Support: 0.1345 (13.45%) | Count: 16 readers

Itemset: PROBABILITES : RAPPELS DE COURS ET EXERCICES CORRIGES
  Support: 0.1092 (10.92%) | Count: 13 readers

Itemset: FONCTIONS DE PLUSIEURS VARIABLES RELLES : IMITES,CONTINUITE,DIFFERENTIABILITÉ ET.... COURS DÉTAILLÉ ET EXERCICES RÉSOLUS, PROBABILITES : RAPPELS DE COURS ET EXERCICES CORRIGES
  Support: 0.0924 (9.24%) | Count: 11 readers

Itemset: MATHÉMATIQUES R

In [43]:
print(f"\n{'='*140}")
print("Top 20 Association Rules (in detail):")
print(f"{'='*140}\n")

# Print rules in format: antecedent -> consequent
for idx, (i, row) in enumerate(rules_sorted.head(20).iterrows(), 1):
    rule = f"{row['antecedent_str']} → {row['consequent_str']}"
    print(f"{idx}. {rule}")
    print(f"   Support: {row['support']:.4f} | Confidence: {row['confidence']:.4f} | Lift: {row['lift']:.2f} | Leverage: {row['leverage']:.4f} | Conviction: {row['conviction']:.2f}\n")


print(f"\nOverall Rule Statistics:")
print(f"  Average confidence: {rules['confidence'].mean():.4f}")
print(f"  Average support: {rules['support'].mean():.4f}")
print(f"  Average lift: {rules['lift'].mean():.2f}")
print(f"  Average leverage: {rules['leverage'].mean():.4f}")
print(f"  Average conviction: {rules['conviction'].mean():.2f}")


Top 20 Association Rules (in detail):

1. PROBABILITES : RAPPELS DE COURS ET EXERCICES CORRIGES → FONCTIONS DE PLUSIEURS VARIABLES RELLES : IMITES,CONTINUITE,DIFFERENTIABILITÉ ET.... COURS DÉTAILLÉ ET EXERCICES RÉSOLUS
   Support: 0.0924 | Confidence: 0.8462 | Lift: 4.03 | Leverage: 0.0695 | Conviction: 5.13

2. EXERCICES EN LANGAGE C++ : 178 EXERCICES CORRIGÉS → ALGEBRE 1 : RAPPELS DE COURS ET EXERCICES AVEC SOLUTIONS
   Support: 0.0252 | Confidence: 0.7500 | Lift: 2.35 | Leverage: 0.0145 | Conviction: 2.72

3. TOUT SUR R : ENSEMBLE DES NOMBRES REELS STRUCTURES ALGEBRIQUE ET TOPOLOGIQUE → ALGEBRE 1 : RAPPELS DE COURS ET EXERCICES AVEC SOLUTIONS
   Support: 0.0420 | Confidence: 0.7143 | Lift: 2.24 | Leverage: 0.0232 | Conviction: 2.38

4. ELÉMENTS DE LA THÉORIE DES PROBABILITÉS T.1 → FONCTIONS DE PLUSIEURS VARIABLES RELLES : IMITES,CONTINUITE,DIFFERENTIABILITÉ ET.... COURS DÉTAILLÉ ET EXERCICES RÉSOLUS
   Support: 0.0252 | Confidence: 0.6000 | Lift: 2.86 | Leverage: 0.0164 | Convictio